In [58]:
!pip install finrl
!pip install gymnasium
!pip install stable-baselines3
!pip install yfinance
!pip install alpaca-trade-api
!pip install exchange_calendars
!pip install stockstats


In [64]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import yfinance as yf


###### Define Custom Trading Environment: A custom trading environment is 
###### created using `gymnasium`. This environment simulates stock trading by allowing actions 
###### like Buy, Hold, and Sell. It also provides observations (stock data) and calculates rewards
###### based on the actions taken.

In [85]:
class CustomTradingEnv(gym.Env):
    def __init__(self, df):
        super(CustomTradingEnv, self).__init__()
        self.df = df
        self.current_step = 0
        self.action_space = spaces.Discrete(3)  # Buy, Hold, Sell
        self.observation_space = spaces.Box(low=0, high=1, shape=(len(df.columns),), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        return self._next_observation(), {}

    def _next_observation(self):
        return self.df.iloc[self.current_step].values

    def step(self, action):
        self.current_step += 1
        reward = 0  # Define your reward function here
        done = self.current_step >= len(self.df) - 1
        obs = self._next_observation()
        info = {}
        return obs, reward, done, False, info

    def render(self, mode='human'):
        pass


##### Fetch historical stock data
##### The code fetches historical stock data for a specific ticker (e.g., AAPL) using 
##### `yfinance`. The data includes columns like Open, High, Low, Close, and Volume.

In [68]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2020-01-01"
df = yf.download(ticker, start=start_date, end=end_date)
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df = df.pct_change().dropna()

# Create the environment
env = CustomTradingEnv(df)

[*********************100%***********************]  1 of 1 completed


##### Train the agent
##### A reinforcement learning agent is trained using the custom trading environment. 
##### The agent learns to make trading decisions based on the stock data.


In [72]:
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
env = make_vec_env(lambda: CustomTradingEnv(df), n_envs=1)
model = A2C('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=10000)
model.save("a2c_trading_model")




Using cpu device
------------------------------------
| time/                 |          |
|    fps                | 303      |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -1.1     |
|    explained_variance | -115     |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -0.0022  |
|    value_loss         | 6.01e-06 |
------------------------------------
------------------------------------
| time/                 |          |
|    fps                | 314      |
|    iterations         | 200      |
|    time_elapsed       | 3        |
|    total_timesteps    | 1000     |
| train/                |          |
|    entropy_loss       | -1.1     |
|    explained_variance | -92.6    |
|    learning_rate      | 0.0007   |
|    n_updates          | 199      |
|    policy_loss        | 0.000849 |
|    value_loss      

In [74]:
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env

# Create the environment
env = make_vec_env(lambda: CustomTradingEnv(df), n_envs=1)

# Train the agent
model = A2C('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=10000)

# Save the model
model.save("a2c_trading_model")


Using cpu device
------------------------------------
| time/                 |          |
|    fps                | 291      |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -1.09    |
|    explained_variance | -29.9    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -0.00222 |
|    value_loss         | 9.67e-06 |
------------------------------------
-------------------------------------
| time/                 |           |
|    fps                | 305       |
|    iterations         | 200       |
|    time_elapsed       | 3         |
|    total_timesteps    | 1000      |
| train/                |           |
|    entropy_loss       | -1.1      |
|    explained_variance | -60.1     |
|    learning_rate      | 0.0007    |
|    n_updates          | 199       |
|    policy_loss        | -0.000587 |
|    valu

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Create the environment
env = make_vec_env(lambda: CustomTradingEnv(df), n_envs=1)

# Train the agent
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=10000)

# Save the model
model.save("ppo_trading_model")


In [ ]:
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
import numpy as np

# Create the environment
env = make_vec_env(lambda: CustomTradingEnv(df), n_envs=1)

# Add some noise for exploration
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Train the agent
model = DDPG('MlpPolicy', env, action_noise=action_noise, verbose=1)
model.learn(total_timesteps=10000)

# Save the model
model.save("ddpg_trading_model")
